In [4]:
import jax.numpy as jnp
from jax import vmap
import numpy as np

class PiecewiseLinearPolynomialsHex:
    def __init__(self, N_t, N_x, N_y, t_min=0.0, t_max=1.0, x_min=-1.0, x_max=1.0, y_min=-1.0, y_max=1.0, gauss_order=3, bdry_order=3):
        """
        Initialize piecewise linear polynomials on a hexahedral mesh with custom ranges.

        Args:
            N_t: Number of divisions along the t-axis.
            N_x: Number of divisions along the x-axis.
            N_y: Number of divisions along the y-axis.
            t_min: Minimum value for t (default: 0.0).
            t_max: Maximum value for t (default: 1.0).
            x_min: Minimum value for x (default: -1.0).
            x_max: Maximum value for x (default: 1.0).
            y_min: Minimum value for y (default: -1.0).
            y_max: Maximum value for y (default: 1.0).
            gauss_order: Order of Gauss quadrature (number of points per direction, default: 3).
            bdry_order: Order of boundary quadrature (number of points per face direction, default: 3).
        """
        self.N_t = N_t
        self.N_x = N_x
        self.N_y = N_y
        self.t_min = t_min
        self.t_max = t_max
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.gauss_order = gauss_order
        self.bdry_order = bdry_order
        self.N_elements = N_t * N_x * N_y  # Number of hexahedra
        self.partitions = self._generate_hexahedra()
        self.coeffs = jnp.zeros((self.N_elements, 4))  # Coefficients a + b*t + c*x + d*y per hexahedron
        self.quad_points, self.quad_weights, self.element_indices, self.quad_points_parti, self.quad_weights_parti = self._setup_quadrature()
        # self.bdry_points_parti, self.bdry_weights_parti = self.get_boundary_quadrature()
        # self.normals = self.compute_normals()

    def _generate_hexahedra(self):
        """Generate the hexahedra for the mesh with custom ranges."""
        self.tau = (self.t_max - self.t_min) / self.N_t
        self.hx = (self.x_max - self.x_min) / self.N_x
        self.hy = (self.y_max - self.y_min) / self.N_y
        t_starts = jnp.linspace(self.t_min, self.t_max - self.tau, self.N_t)
        x_starts = jnp.linspace(self.x_min, self.x_max - self.hx, self.N_x)
        y_starts = jnp.linspace(self.y_min, self.y_max - self.hy, self.N_y)
        tl, xl, yl = jnp.meshgrid(t_starts, x_starts, y_starts, indexing='ij')
        # Define the 8 vertices for each hexahedron using binary offsets
        offsets_bin = jnp.array([[int(b) for b in f'{i:03b}'] for i in range(8)], dtype=jnp.float32)  # (8,3) binary 000 to 111
        deltas = jnp.array([self.tau, self.hx, self.hy])
        offsets = offsets_bin * deltas  # (8,3)
        lowers = jnp.stack([tl.ravel(), xl.ravel(), yl.ravel()], axis=1)[:, None, :]  # (N_elements, 1, 3)
        partitions = lowers + offsets[None, :, :]  # (N_elements, 8, 3)
        return partitions
        
    def _setup_quadrature(self):
        """
        Setup Gauss quadrature points and weights for hexahedral mesh.

        Returns:
            quad_points: Quadrature points, shape (N_quad, 3).
            quad_weights: Quadrature weights, shape (N_quad,).
            element_indices: Indices of hexahedra, shape (N_quad,).
            quad_points_parti: Quadrature points per hexahedron, shape (N_elements, Nq, 3).
            quad_weights_parti: Quadrature weights per hexahedron, shape (N_elements, Nq).
        """
        Nq = self.gauss_order ** 3  # Number of quadrature points per hexahedron

        # Gauss-Legendre nodes and weights in [-1, 1]
        nodes, weights = np.polynomial.legendre.leggauss(self.gauss_order)
        nodes_jnp = jnp.array(nodes)
        weights_jnp = jnp.array(weights)

        # Tensor product for 3D
        t_nodes, x_nodes, y_nodes = jnp.meshgrid(nodes_jnp, nodes_jnp, nodes_jnp)
        t_weights, x_weights, y_weights = jnp.meshgrid(weights_jnp, weights_jnp, weights_jnp)
        quad_weights_3d = (t_weights * x_weights * y_weights).ravel()

        # Function to map [-1,1]^3 to hexahedron [t_low, t_high]x[x_low, x_high]x[y_low, y_high]
        def map_to_hex(hex):
            t_low, x_low, y_low = hex[0]
            t_high, x_high, y_high = hex[-1]
            t_scale = (t_high - t_low) / 2
            x_scale = (x_high - x_low) / 2
            y_scale = (y_high - y_low) / 2
            t_center = (t_high + t_low) / 2
            x_center = (x_high + x_low) / 2
            y_center = (y_high + y_low) / 2
            t_points = t_center + t_nodes.ravel() * t_scale
            x_points = x_center + x_nodes.ravel() * x_scale
            y_points = y_center + y_nodes.ravel() * y_scale
            points = jnp.stack([t_points, x_points, y_points], axis=1)
            weights = quad_weights_3d * (t_scale * x_scale * y_scale)
            return points, weights

        # Vectorize over hexahedra
        quad_points_parti, quad_weights_parti = vmap(map_to_hex)(self.partitions)

        # Flatten for global quadrature
        quad_points = quad_points_parti.reshape(-1, 3)
        quad_weights = quad_weights_parti.reshape(-1)
        element_indices = jnp.repeat(jnp.arange(self.N_elements), Nq)

        return quad_points, quad_weights, element_indices, quad_points_parti, quad_weights_parti


In [15]:
import jax.numpy as jnp
from jax import vmap
import numpy as np
from scipy.integrate import tplquad
import math

# Define a complicated test function (non-polynomial to stress the quadrature)
def test_function(t, x, y):
    return jnp.sin(2 * jnp.pi * t) * jnp.cos(3 * jnp.pi * x) * jnp.exp(y) + t * x**2 * y**3 + jnp.log(1 + jnp.abs(t + x + y))

# Vectorized version for JAX quadrature points
def test_function_vec(points):
    t, x, y = points[:, 0], points[:, 1], points[:, 2]
    return test_function(t, x, y)

# SciPy version (uses NumPy)
def test_function_scipy(y, x, t):  # Order for tplquad: integrand(y, x, t) with dy dx dt
    return np.sin(2 * np.pi * t) * np.cos(3 * np.pi * x) * np.exp(y) + t * x**2 * y**3 + np.log(1 + np.abs(t + x + y))

# Function to compute quadrature integral
def compute_quadrature_integral(plph):
    return jnp.sum(test_function_vec(plph.quad_points) * plph.quad_weights)

# Function to compute SciPy integral (high accuracy)
def compute_scipy_integral(t_min, t_max, x_min, x_max, y_min, y_max):
    integral, error = tplquad(
        test_function_scipy,
        t_min, t_max,  # dt
        lambda t: x_min, lambda t: x_max,  # dx
        lambda t, x: y_min, lambda t, x: y_max,  # dy
        epsabs=1e-10, epsrel=1e-10
    )
    return integral, error

# Basic check: sum of weights should equal domain volume
def check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max):
    computed_volume = jnp.sum(plph.quad_weights)
    exact_volume = (t_max - t_min) * (x_max - x_min) * (y_max - y_min)
    print(f"Computed volume: {computed_volume}, Exact volume: {exact_volume}")
    return abs(computed_volume - exact_volume) < 1e-10

# Test with specific parameters
N_t, N_x, N_y = 3, 3, 3  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
gauss_order = 5  # Default

plph = PiecewiseLinearPolynomialsHex(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, gauss_order)

# Check volume
volume_check = check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max)
print(f"Volume check passed: {volume_check}")

# Compute integrals
quad_int = compute_quadrature_integral(plph)
# scipy_int, scipy_error = compute_scipy_integral(t_min, t_max, x_min, x_max, y_min, y_max)

print(f"Quadrature integral: {quad_int}")
print(f"SciPy integral: {scipy_int} (estimated error: {scipy_error})")

# Compare (difference should be small, depending on gauss_order and mesh refinement)
difference = abs(quad_int - scipy_int)
print(f"Absolute difference: {difference}")
print(f"Relative difference: {difference / abs(scipy_int) if scipy_int != 0 else 'N/A'}")

# To further test correctness, you can increase gauss_order, N_t, N_x, N_y and observe convergence to SciPy value.2.9s

Computed volume: 4.0, Exact volume: 4.0
Volume check passed: True
Quadrature integral: 2.181187391281128
SciPy integral: 2.181217770752753 (estimated error: 5.273749182494284e-07)
Absolute difference: 3.0279159545898438e-05
Relative difference: 1.3881768609280698e-05


In [12]:
# Test with specific parameters
N_t, N_x, N_y = 3, 3, 3  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
gauss_order = 3  # Default

plph = PiecewiseLinearPolynomialsHex(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, gauss_order)

# Check volume
volume_check = check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max)
print(f"Volume check passed: {volume_check}")

# Compute integrals
quad_int = compute_quadrature_integral(plph)

Computed volume: 3.999999761581421, Exact volume: 4.0
Volume check passed: False


In [16]:
plph.quad_weights.shape

(3375,)

In [17]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu
from seq_minmax_opt_class import SeqMinMaxOptProb
from minmax_opt_class_mgnn_2d import MinMaxOptProb2D
from my_helpers import plot_comparison_seperate_combined_u_theta, compute_errors_combined_u_theta, plot_u_theta

In [18]:
def u0(x):
    return (-jnp.sign(x) + 1)*2

# Boundary at x=-1
def u1(x):
    return 4
vmap_u1 = vmap(u1, in_axes=0, out_axes=0)

# Boundary at x=2
def u2(x):
    return 0
vmap_u2 = vmap(u2, in_axes=0, out_axes=0)

def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x

def u_ref(t, x):
    return u0(x-2*t)

t_min = 0
t_max = 0.5
x_min = -0.5
x_max = 1.5


In [20]:
N_t = 8
N_x = 8
seq_trainer = SeqMinMaxOptProb(
    input_dim=2, num_segments=3,
    total_t_min=t_min, total_t_max=t_max, x_min=x_min, x_max=x_max,
    N_t=N_t, N_x=N_x, N_bc=2048, N_ic=4096,
    hidden_dims_list=[[64, 64, 64, 64, 64]], seed=0, pre_epochs=0,
    sigma=5, N_perturb=20000, k_h_rf_times=2, f_opt_kh=1,
    u0=u0, u1=vmap_u1, u2=vmap_u2, project="minmax_burgers_strong_moving_shock",
    gamma=0, N_test_gamma=1000, adjust_gamma=False, lambda_ic=10, lambda_bc=10, lambda_eqn=1,
    L1reg=0, sampling_method='uniform', notes='', 
    init_lrs=[1e-3, 1e-3, 1e-3], grade_epochs=[5000, 1000, 1000], decay_rates=[1e-3, 1e-3, 1e-3],
    res=res, loss_type='interior', k=0, f_plot=1000, partition='tri', 
    set_wandb=False
)


In [ ]:
# This result is by 32*64 mesh
seq_trainer.train_seq()
plot_comparison_seperate_combined_u_theta(seq_trainer, u_ref=u_ref, x_min=-0.5, x_max=1.5, ts=[0.1*i for i in range(6)])
compute_errors_combined_u_theta(seq_trainer, u_ref=u_ref, x_min=-0.5, x_max=1.5, T=0.5)